# Assignment 08 — Grouped Query Attention from Scratch (100 points)

This assignment mirrors **USAAIO 2025 Round 2 Problem 2, Parts 6-8**. You will implement GQA with broadcasting (no loops), prove rank properties of GQA weight matrices, and prove that MHA is a special case of GQA.

**Notation:**
- $B$: batch size, $L$: sequence length, $D$: model dimension
- $H$: number of query heads
- $G$: number of KV groups ($G | H$, i.e., $G$ divides $H$)
- $D_{qk}$: per-head query/key dimension, $D_v$: per-head value dimension
- $W^Q_h \in \mathbb{R}^{D \times D_{qk}}$: per-head query projection ($h = 1, \ldots, H$)
- $W^K_g \in \mathbb{R}^{D \times D_{qk}}$: per-group key projection ($g = 1, \ldots, G$)
- $W^V_g \in \mathbb{R}^{D \times D_v}$: per-group value projection ($g = 1, \ldots, G$)

In GQA, query heads $\{(g-1)H/G + 1, \ldots, gH/G\}$ share key/value from group $g$.

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

**WARNING**: You may only use `torch`, `torch.nn`, `torch.nn.functional`, and `numpy`. No other imports are allowed.

## Part 1 (10 points, non-coding task)

Given $D = 512, H = 8, G = 2, D_{qk} = 64, D_v = 64$.

1. How many query heads per group? (2 points)
2. Shape of the concatenated query projection $W^Q \in \mathbb{R}^{D \times ?}$. (2 points)
3. Shape of the concatenated key projection $W^K \in \mathbb{R}^{D \times ?}$. Note: only $G$ groups. (2 points)
4. Shape of the concatenated value projection $W^V$. (2 points)
5. Compare parameter count for GQA vs. MHA (just K and V projections). (2 points)

Reasoning is required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 2 (10 points, non-coding task)

Describe the tensor reshaping strategy for GQA (no loops).

With $B=2, L=10, H=8, G=2, D_{qk}=64$:

1. After projecting Q: shape $(B, L, H \cdot D_{qk})$. Write the shape. (1 point)
2. Reshape Q to expose group structure: $(B, L, G, H/G, D_{qk})$. Write the shape. (2 points)
3. Permute Q to $(B, G, H/G, L, D_{qk})$. Write the shape. (2 points)
4. After projecting K: $(B, L, G \cdot D_{qk})$. Reshape to $(B, G, L, D_{qk})$. Then unsqueeze to $(B, G, 1, L, D_{qk})$. Show each step. (3 points)
5. Explain how broadcasting handles the `Q @ K.mT` computation with shapes $(B, G, H/G, L, D_{qk})$ and $(B, G, 1, L, D_{qk})$. (2 points)

Reasoning is required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 3 (20 points, coding task)

Implement `MyGQA` as an `nn.Module`. **NO LOOPS.**

Constructor: `MyGQA(D, H, G, D_qk, D_v)`
- `D`: model dimension
- `H`: number of query heads
- `G`: number of KV groups (asserts `H % G == 0`)
- `D_qk`: per-head query/key dimension
- `D_v`: per-head value dimension

Projections:
- `W_Q`: `nn.Linear(D, H * D_qk, bias=False)` — $H$ query heads
- `W_K`: `nn.Linear(D, G * D_qk, bias=False)` — $G$ key groups
- `W_V`: `nn.Linear(D, G * D_v, bias=False)` — $G$ value groups
- `W_O`: `nn.Linear(H * D_v, D, bias=False)` — output projection

Forward: `forward(X)` where $X: (B, L, D)$, returns $(B, L, D)$.

Use broadcasting via `unsqueeze` on K and V. Annotate shapes after EVERY operation.

Reasoning is not required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 4 (10 points, coding task)

Test `MyGQA`.

1. Create with $D=64, H=8, G=2, D_{qk}=16, D_v=16$.
2. Random input $(2, 10, 64)$.
3. Assert output shape $(2, 10, 64)$.
4. Print parameter count. Compare with MHA ($G=H=8$): create a second instance and compare.
5. Verify the parameter reduction: GQA K+V params should be $G/H$ of MHA K+V params.

Reasoning is not required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 5 (10 points, non-coding task)

**Proof: Rank of repeated GQA weight matrix.**

In GQA, if we "unroll" the grouping to look like MHA, each group's $W^K_g \in \mathbb{R}^{D \times D_{qk}}$ is repeated $H/G$ times to form:

$$\tilde{W}^K = [W^K_g \;|\; W^K_g \;|\; \cdots \;|\; W^K_g] \in \mathbb{R}^{D \times (H/G \cdot D_{qk})}$$

Prove that $\text{rank}(\tilde{W}^K) = \text{rank}(W^K_g)$.

Your proof must:
1. Define rank in terms of column space or linearly independent columns. (2 points)
2. Show that the column space of $\tilde{W}^K$ equals the column space of $W^K_g$. (5 points)
3. Conclude the ranks are equal. (3 points)

Reasoning is required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 6 (5 points, coding task)

Verify the rank proof numerically.

1. Create a random matrix $W \in \mathbb{R}^{64 \times 16}$.
2. Repeat it 4 times: $\tilde{W} = [W | W | W | W] \in \mathbb{R}^{64 \times 64}$.
3. Compute rank of $W$ and $\tilde{W}$ using `torch.linalg.matrix_rank`.
4. Assert they are equal.
5. Compare with a random $\hat{W} \in \mathbb{R}^{64 \times 64}$ (all columns independent). What is its rank?

Reasoning is not required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 7 (10 points, non-coding task)

**Proof: MHA is a special case of GQA.**

Prove that Multi-Head Attention is a special case of Grouped Query Attention.

Your proof must:
1. State what value of $G$ makes GQA equivalent to MHA. (2 points)
2. Show that with this $G$, each group has exactly one query head. (3 points)
3. Show that the GQA computation reduces to the standard MHA computation term by term. (5 points)

Reasoning is required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 8 (10 points, coding task)

Verify numerically that GQA with $G = H$ produces the same output as MHA.

1. Create `MyGQA(D=32, H=4, G=4, D_qk=8, D_v=8)` — this should be MHA.
2. Create a standard `MyMHA(D1=32, D2=32, H=4, D_qk=8, D_v=8)` from Assignment 03.
3. Copy the weights from GQA to MHA (they should have the same shapes when $G = H$).
4. Pass the same random input through both.
5. Assert outputs are equal (within tolerance).

Reasoning is not required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 9 (10 points, coding task)

KV-cache analysis for GQA.

For $D = 4096, H = 32, D_{qk} = 128, 80$ layers, FP16 (2 bytes):

1. Compute the KV-cache per position per layer for $G \in \{32, 8, 4, 1\}$. (4 points)
2. Compute total cache for $L = 8192$ positions across all 80 layers. Express in GB. (4 points)
3. Print a table comparing all configurations. (2 points)

Reasoning is not required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 10 (5 points, non-coding task)

Multi-Query Attention (MQA) uses $G = 1$.

1. What is the KV-cache reduction of MQA vs. MHA? Express as a factor. (2 points)
2. MQA has been observed to sometimes reduce model quality. Explain intuitively why sharing ALL K/V across query heads might be harmful. (3 points)

Reasoning is required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """